# v4: optimizing the int8 counter before renting a GPU

v2 and v3 showed that an int8 flip counter (8 bits per weight) trains a ternary model as well as fp32 latent weights + Adam (96 bits), with STE gradients. Before paying for the SmolLM2-135M conversion, this notebook tries to make the counter better and checks the real training script.

| stage | what | time on Kaggle |
|---|---|---|
| 1 | Counter variants (τ, reset rule, leak, per-row votes, vote clipping, τ schedules) vs a stronger latent baseline with LR decay, from scratch on the toy model; screen, then confirm the best with 3 seeds | ~35 min |
| 2 | Toy **conversion**: pretrain an fp toy model, then convert it to ternary. Latent + λ warmup (Hugging Face recipe) vs counter, with and without counters initialized from the fp weights (κ) | ~10 min |
| 3 | **SmolLM2-135M pilot** with `train_bitnet.py`, the same script you will run on the rented GPU: data prep, arms 0 and 0b, then latent (GPU 0) and counter (GPU 1) in parallel | ~2–3 h |

Settings: **GPU T4 ×2**, internet **on**. Stages 1–2 need one GPU; stage 3 uses both.

In [ ]:
import math, time, os, glob, json, urllib.request, sysconfig
from contextlib import nullcontext
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('device:', dev, torch.cuda.get_device_name(0) if dev == 'cuda' else '')

CFG   = dict(V=256, d=128, L=4, H=4, T=128)
B     = 8
CHUNK = 128 if dev == 'cuda' else 16
QUANT = dict(on=True, act_bits=8)
USE_FP16 = False      # set automatically by the fp16 check in Phase B
FP_LR = 3e-3          # Adam lr for embeddings / head (all rules)
BETA_LR = 3e-4        # Adam lr for the per-matrix scale beta (ternary-native rules)

## Data (tiny Shakespeare, byte-level) and the toy BitNet model, as in v2

In [ ]:
def load_text():
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    try:
        raw = urllib.request.urlopen(url, timeout=20).read(); print('tiny shakespeare,', len(raw), 'bytes'); return raw
    except Exception as e:
        print('download failed, using Python stdlib source:', e)
        files = sorted(glob.glob(os.path.join(sysconfig.get_paths()['stdlib'], '*.py')))
        return b''.join(open(f, 'rb').read() for f in files)[:3_000_000]

data = torch.from_numpy(np.frombuffer(load_text(), dtype=np.uint8).astype(np.int64))
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

def get_batch(split, gen, bs=B, T=CFG['T']):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - T - 1, (bs,), generator=gen)
    x = torch.stack([d[i:i + T] for i in ix]); y = torch.stack([d[i + 1:i + T + 1] for i in ix])
    return x.to(dev), y.to(dev)

In [ ]:
def ste(x, q): return q.detach() + (x - x.detach())   # forward = exactly q, gradient = identity
def rms(x):    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)

def act_quant(x):
    if not QUANT['on']: return x
    Qp = 2 ** (QUANT['act_bits'] - 1) - 1
    s = Qp / x.abs().amax(-1, keepdim=True).clamp(min=1e-5)
    return ste(x, (x * s).round().clamp(-Qp - 1, Qp) / s)

def weight_quant(w):
    if not QUANT['on']: return w
    beta = w.abs().mean().clamp(min=1e-5)
    return ste(w, (w / beta).round().clamp(-1, 1) * beta)

class BitLinear(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dout, din) / math.sqrt(din))
        self.native = False
    @torch.no_grad()
    def to_native(self):
        b = self.weight.abs().mean().clamp(min=1e-5)
        self.register_buffer('t', (self.weight / b).round().clamp(-1, 1).to(torch.int8))
        self.beta = nn.Parameter(b.clone())
        self.weight.requires_grad_(False)          # latent no longer used
        self.native = True
    def forward(self, x, cache=None, name=None):
        xq = act_quant(rms(x))
        if cache is not None: cache[name] = xq.detach()
        if self.native:
            w = self.beta * self.t.float()
            if w.requires_grad: w.retain_grad(); self._w = w
        else:
            w = weight_quant(self.weight)
        return F.linear(xq, w)

def attention(q, k, v, H):
    N, T, D = q.shape; hd = D // H
    sh = lambda t: t.reshape(N, T, H, hd).transpose(1, 2)
    o = F.scaled_dot_product_attention(sh(q), sh(k), sh(v), is_causal=True)
    return o.transpose(1, 2).reshape(N, T, D)

class Block(nn.Module):
    def __init__(self, d, H):
        super().__init__()
        self.H = H
        self.wq, self.wk, self.wv, self.wo = (BitLinear(d, d) for _ in range(4))
        self.w1, self.w2 = BitLinear(d, 4 * d), BitLinear(4 * d, d)
    def forward(self, h, noise=None, cache=None, p=''):
        nz = noise or {}
        add = lambda name, t: (t + nz[name]) if name in nz else t
        q = add('q', self.wq(h, cache, p + 'wq'))
        k = add('k', self.wk(h, cache, p + 'wk'))
        v = add('v', self.wv(h, cache, p + 'wv'))
        o = add('attn_o', attention(q, k, v, self.H))
        if cache is not None:
            cache[p + 'q'], cache[p + 'k'], cache[p + 'v'], cache[p + 'o'] = q.detach(), k.detach(), v.detach(), o.detach()
        h = h + add('wo', self.wo(o, cache, p + 'wo'))
        u = add('w1', self.w1(h, cache, p + 'w1'))
        h = h + add('w2', self.w2(F.relu(u).square(), cache, p + 'w2'))
        return h

class BitGPT(nn.Module):
    def __init__(self, V, d, L, H, T):
        super().__init__()
        self.L = L
        self.tok  = nn.Parameter(torch.randn(V, d))
        self.pos  = nn.Parameter(torch.randn(T, d) * 0.5)
        self.blocks = nn.ModuleList(Block(d, H) for _ in range(L))
        self.head = nn.Parameter(torch.randn(V, d) / math.sqrt(d))
    def embed(self, idx): return self.tok[idx] + self.pos[: idx.shape[1]]
    def run(self, h, start=0, nb=None, noise=None, cache=None):
        for i in range(start, self.L):
            if cache is not None: cache[f'h{i}'] = h.detach()
            h = self.blocks[i](h, noise if i == nb else None, cache, f'b{i}.')
        hf = rms(h)
        if cache is not None: cache['hf'] = hf.detach()
        return hf @ self.head.t().to(hf.dtype)
    def forward(self, idx, cache=None):
        e = self.embed(idx)
        if cache is not None: cache['e'] = e.detach()
        return self.run(e, cache=cache)

def tok_loss(logits, y):
    return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), y.reshape(-1), reduction='none').view(y.shape)

def bitlinears(model):
    return {f'b{i}.{n}': getattr(b, n) for i, b in enumerate(model.blocks) for n in ['wq', 'wk', 'wv', 'wo', 'w1', 'w2']}

def make_model(update, seed=0):
    torch.manual_seed(seed)
    m = BitGPT(**CFG).to(dev)
    if update != 'latent':
        for mod in bitlinears(m).values(): mod.to_native()
    return m

print('params:', sum(p.numel() for p in make_model('latent').parameters()))

In [ ]:
SIGMA = dict(embed=0.1, q=0.1, k=0.1, v=0.1, attn_o=0.1, wo=0.1, w1=0.1, w2=0.1, head=0.3)
GAMMA = dict(loss=0.0, q=0.0, k=0.98, v=0.98)

def amp():
    return torch.autocast('cuda', dtype=torch.float16) if (USE_FP16 and dev == 'cuda') else nullcontext()

def discount(T, g):
    t = torch.arange(T, device=dev)
    diff = (t[None, :] - t[:, None]).float()
    return torch.where(diff >= 0, g ** diff.clamp(min=0), torch.zeros_like(diff))

@torch.no_grad()
def estimate_out_err(loss_fn, shape, sigma, K, chunk, D):
    g = torch.zeros(tuple(shape), device=dev); done = 0
    while done < K:
        Kc = max(2, min(chunk, K - done))
        a = torch.randn((Kc,) + tuple(shape), device=dev)
        with amp():
            l = loss_fn(sigma * a).float()
        c = l.mean(0, keepdim=True) - l
        r = c @ D.t()
        g += torch.einsum('kbt,kbtd->btd', r, a)
        done += Kc
    return -g / (done * sigma)

@torch.no_grad()
def local_attn_err(cache, p, g_o, site, H, sigma, K, chunk, D):
    q0, k0, v0, o0 = (cache[p + n] for n in ['q', 'k', 'v', 'o'])
    Bb, T, d = q0.shape; hd = d // H
    g = torch.zeros(Bb, T, H, hd, device=dev); done = 0
    while done < K:
        Kc = max(2, min(chunk, K - done))
        a = torch.randn(Kc, Bb, T, d, device=dev)
        qkv = {n: t.unsqueeze(0).expand(Kc, Bb, T, d) for n, t in zip('qkv', (q0, k0, v0))}
        qkv[site] = qkv[site] + sigma * a
        with amp():
            o = attention(*(qkv[n].reshape(Kc * Bb, T, d) for n in 'qkv'), H)
        o = o.float().view(Kc, Bb, T, d)
        c = -((o - o0) * g_o).view(Kc, Bb, T, H, hd).sum(-1)
        c = c - c.mean(0, keepdim=True)
        r = torch.einsum('kbsh,ts->kbth', c, D)
        g += torch.einsum('kbth,kbthe->bthe', r, a.view(Kc, Bb, T, H, hd))
        done += Kc
    return -g.reshape(Bb, T, d) / (done * sigma)

@torch.no_grad()
def dust_grads(model, x, y, K=256, K_head=None, chunk=CHUNK, sigma=None, gamma=None):
    sigma = sigma or SIGMA; gamma = gamma or GAMMA
    Bb, T = x.shape; N = Bb * T; H = model.blocks[0].H
    cache = {}; model(x, cache=cache)
    Dl = discount(T, gamma['loss'])
    yK = lambda Kc: y.repeat(Kc, 1)
    outer = lambda g, xin: torch.einsum('bto,bti->oi', g, xin) / N
    G = {}
    hf = cache['hf']; logits = hf @ model.head.t()
    def f_head(nz):
        Kc = nz.shape[0]
        return tok_loss((logits.unsqueeze(0) + nz).reshape(Kc * Bb, T, -1), yK(Kc)).view(Kc, Bb, T)
    G['head'] = outer(estimate_out_err(f_head, logits.shape, sigma['head'], K_head or 4 * K, chunk, Dl), hf)
    e = cache['e']
    def f_emb(nz):
        Kc = nz.shape[0]
        return tok_loss(model.run((e.unsqueeze(0) + nz).reshape(Kc * Bb, T, -1)), yK(Kc)).view(Kc, Bb, T)
    g = estimate_out_err(f_emb, e.shape, sigma['embed'], K, chunk, Dl)
    G['tok'] = torch.zeros_like(model.tok).index_add_(0, x.reshape(-1), g.reshape(N, -1)) / N
    G['pos'] = torch.zeros_like(model.pos); G['pos'][:T] = g.sum(0) / N
    for l in range(model.L):
        p = f'b{l}.'; d = cache[p + 'q'].shape[-1]; h0 = cache[f'h{l}']
        def f_site(site):
            def f(nz):
                Kc = nz.shape[0]
                lg = model.run(h0.repeat(Kc, 1, 1), start=l, nb=l, noise={site: nz.reshape(Kc * Bb, T, -1)})
                return tok_loss(lg, yK(Kc)).view(Kc, Bb, T)
            return f
        for site, dout in [('wo', d), ('w1', 4 * d), ('w2', d)]:
            G[p + site] = outer(estimate_out_err(f_site(site), (Bb, T, dout), sigma[site], K, chunk, Dl), cache[p + site])
        g_o = estimate_out_err(f_site('attn_o'), (Bb, T, d), sigma['attn_o'], K, chunk, Dl)
        for site in 'qkv':
            g = local_attn_err(cache, p, g_o, site, H, sigma[site], K, chunk, discount(T, gamma[site]))
            G[p + 'w' + site] = outer(g, cache[p + 'w' + site])
    return G

def ste_grads(model, x, y):
    model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        loss = tok_loss(model(x), y).mean(); loss.backward()
    G = {n: (m._w.grad if m.native else m.weight.grad).detach().clone() for n, m in bitlinears(model).items()}
    for n in ['tok', 'pos', 'head']: G[n] = getattr(model, n).grad.detach().clone()
    return G, loss.item()

def mean_cos(Ga, Gb):
    return float(np.mean([F.cosine_similarity(Ga[k].flatten(), Gb[k].flatten(), dim=0).item() for k in Gb]))

## Counter variants, λ warmup and the training loop

`hp` keys for the counter:
- `tau`: threshold. `anneal`: τ multiplier reached at the end (an LR-decay analogue). `ramp`: fraction of training over which τ ramps up from τ/4.
- `reset`: `'zero'` (v2) or `'sub'` (keep the excess vote after a flip).
- `leak`: per-step decay of the counter toward 0 (forgets stale votes).
- `norm`: votes normalized per matrix (`'mat'`, v2) or per output row (`'row'`).
- `clip`: cap on a single vote.
- `kappa`: when converting a pretrained model, initialize each counter from the weight's leftover fraction (how close it was to flipping).

λ blends quantization in (Hugging Face's warmup); evaluation always uses the fully ternary model (λ = 1).

In [ ]:
LAM = dict(v=1.0)

def act_quant(x):
    if not QUANT['on']: return x
    Qp = 2 ** (QUANT['act_bits'] - 1) - 1
    s = Qp / x.abs().amax(-1, keepdim=True).clamp(min=1e-5)
    q = (x * s).round().clamp(-Qp - 1, Qp) / s
    return ste(x, x + LAM['v'] * (q - x))

def weight_quant(w):
    if not QUANT['on']: return w
    beta = w.abs().mean().clamp(min=1e-5)
    q = (w / beta).round().clamp(-1, 1) * beta
    return ste(w, w + LAM['v'] * (q - w))

@torch.no_grad()
def ternary_of(mod):
    if mod.native: return mod.t.clone()
    w = mod.weight; b = w.abs().mean().clamp(min=1e-5)
    return (w / b).round().clamp(-1, 1).to(torch.int8)

def cur_tau(hp, frac):
    tau = hp.get('tau', 32) * (1 + (hp.get('anneal', 1.0) - 1) * frac)
    ramp = hp.get('ramp', 0)
    if ramp and frac < ramp: tau *= 0.25 + 0.75 * frac / ramp
    return tau

@torch.no_grad()
def counter_update_v(mod, G, c, hp, frac):
    tau = cur_tau(hp, frac)
    if hp.get('norm', 'mat') == 'row':
        s = -G / G.pow(2).mean(1, keepdim=True).sqrt().clamp(min=1e-12)
    else:
        s = -G / G.pow(2).mean().sqrt().clamp(min=1e-12)
    if hp.get('clip'): s = s.clamp(-hp['clip'], hp['clip'])
    cf = c.float()
    if hp.get('leak'):
        p = (hp['leak'] * cf.abs()).clamp(max=1)
        cf = cf - torch.sign(cf) * (torch.rand_like(cf) < p).float()
    cf = (cf + torch.floor(s + torch.rand_like(s))).clamp(-127, 127)
    t = mod.t.float(); up, dn = cf >= tau, cf <= -tau
    t_new = torch.where(up, (t + 1).clamp(max=1), torch.where(dn, (t - 1).clamp(min=-1), t))
    if hp.get('reset', 'zero') == 'zero':
        cf = torch.where(up | dn, torch.zeros_like(cf), cf)
    else:
        cf = torch.where(up, cf - tau, torch.where(dn, cf + tau, cf))
    c.copy_(cf.round().to(torch.int8)); mod.t.copy_(t_new.to(torch.int8))

@torch.no_grad()
def native_with_counters(model, hp):
    counters = {}
    for n, m in bitlinears(model).items():
        W = m.weight.data.clone(); b = W.abs().mean().clamp(min=1e-5); u = W / b
        m.to_native()
        tau = hp.get('tau', 32); lim = max(1, tau - 1)
        r = (u - m.t.float()).clamp(-0.5, 0.5)
        counters[n] = (hp.get('kappa', 0.0) * 2 * r * tau).round().clamp(-lim, lim).clamp(-127, 127).to(torch.int8)
    return counters

def make_opt(model, update, hp):
    groups = [{'params': [model.tok, model.pos, model.head], 'lr': FP_LR}]
    BL = bitlinears(model).values()
    if update == 'latent': groups.append({'params': [m.weight for m in BL], 'lr': hp['lr']})
    else:                  groups.append({'params': [m.beta for m in BL], 'lr': BETA_LR})
    return torch.optim.Adam(groups)

def evaluate(model, n=20):
    old = LAM['v']; LAM['v'] = 1.0
    g = torch.Generator().manual_seed(1234); tot = 0.0
    with torch.no_grad():
        for _ in range(n):
            x, y = get_batch('val', g); tot += tok_loss(model(x), y).mean().item()
    LAM['v'] = old
    return tot / n

def train4(update, hp, steps, seed=0, init=None, lam_steps=0, eval_every=250, verbose=False):
    model = make_model('latent', seed)
    if init is not None: model.load_state_dict(init)
    counters = native_with_counters(model, hp) if update == 'counter' else None
    opt = make_opt(model, update, hp)
    sched = None
    if hp.get('sched') == 'cos':
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(s / steps, 1))))
    BL = bitlinears(model); gen = torch.Generator().manual_seed(seed + 100); torch.manual_seed(seed + 7)
    prev = {n: ternary_of(m) for n, m in BL.items()}
    H = dict(step=[], val=[], flips=[], zeros=[]); fl = []; t0 = time.time()
    for step in range(steps + 1):
        LAM['v'] = min(1.0, step / lam_steps) if lam_steps else 1.0
        if step % eval_every == 0 or step == steps:
            tern = [ternary_of(m) for m in BL.values()]
            H['step'].append(step); H['val'].append(evaluate(model))
            H['flips'].append(float(np.mean(fl)) if fl else 0.0)
            H['zeros'].append(sum((t == 0).sum().item() for t in tern) / sum(t.numel() for t in tern)); fl = []
            if verbose: print(f'  step {step:5d} val {H["val"][-1]:.4f} flips {H["flips"][-1]*100:.3f}%')
        if step == steps: break
        x, y = get_batch('train', gen)
        G = ste_grads(model, x, y)[0]
        for n_ in ['tok', 'pos', 'head']: getattr(model, n_).grad = G[n_]
        for n_, m in BL.items():
            if m.native: m.beta.grad = (G[n_] * m.t.float()).sum().reshape(m.beta.shape)
            else:        m.weight.grad = G[n_]
        opt.step()
        if sched: sched.step()
        if update == 'counter':
            for n_, m in BL.items(): counter_update_v(m, G[n_], counters[n_], hp, step / steps)
        nf = tot = 0
        for n_, m in BL.items():
            t = ternary_of(m); nf += (t != prev[n_]).sum().item(); tot += t.numel(); prev[n_] = t
        fl.append(nf / tot)
    LAM['v'] = 1.0
    H['time'] = time.time() - t0
    return model, H

## Stage 1: counter variants from scratch (STE gradients, 2000 steps each)

Each variant changes one thing from the v2 baseline (τ = 32, zero reset). Latent baselines now include cosine LR decay, a fairer, stronger reference than v2's constant LR. One run takes about 70 s.

In [ ]:
QUANT['on'] = True; QUANT['act_bits'] = 8
S1_STEPS = 2000
BASE = dict(tau=32)
VARIANTS = {
    'base (tau 32)':  {},
    'tau 24':         dict(tau=24),
    'tau 40':         dict(tau=40),
    'tau 48':         dict(tau=48),
    'reset sub':      dict(reset='sub'),
    'leak 0.01':      dict(leak=0.01),
    'leak 0.03':      dict(leak=0.03),
    'row norm':       dict(norm='row'),
    'clip 3':         dict(clip=3),
    'anneal x2':      dict(anneal=2.0),
    'ramp 20%':       dict(ramp=0.2),
    'ramp + anneal':  dict(ramp=0.2, anneal=2.0),
}
LATENT = {'latent lr1e-3 const': dict(lr=1e-3), 'latent lr1e-3 cos': dict(lr=1e-3, sched='cos'), 'latent lr3e-3 cos': dict(lr=3e-3, sched='cos')}
s1 = {}
for name, d in VARIANTS.items():
    hp = {**BASE, **d}; _, H = train4('counter', hp, S1_STEPS); s1[name] = (hp, H)
    print(f'{name:22s} val {H["val"][-1]:.4f}  flips/step {np.mean(H["flips"][1:])*100:.3f}%  zeros {H["zeros"][-1]*100:.1f}%  {H["time"]:.0f}s')
for name, hp in LATENT.items():
    _, H = train4('latent', hp, S1_STEPS); s1[name] = (hp, H)
    print(f'{name:22s} val {H["val"][-1]:.4f}  flips/step {np.mean(H["flips"][1:])*100:.3f}%  zeros {H["zeros"][-1]*100:.1f}%  {H["time"]:.0f}s')

### Combine the variants that helped, then confirm with 3 seeds

In [ ]:
base_val = s1['base (tau 32)'][1]['val'][-1]
GROUPS = [['tau 24', 'tau 40', 'tau 48'], ['reset sub'], ['leak 0.01', 'leak 0.03'], ['row norm'], ['clip 3'],
          ['anneal x2', 'ramp 20%', 'ramp + anneal']]
combo = dict(BASE); picked = []
for grp in GROUPS:
    best = min(grp, key=lambda n: s1[n][1]['val'][-1])
    if s1[best][1]['val'][-1] < base_val - 0.01:
        combo.update(VARIANTS[best]); picked.append(best)
best_single = min(VARIANTS, key=lambda n: s1[n][1]['val'][-1])
best_latent = min(LATENT, key=lambda n: s1[n][1]['val'][-1])
print('helped (> 0.01 better than base):', picked or 'none'); print('combo:', combo)

CONFIRM = {'counter base': ('counter', dict(BASE)), f'counter {best_single}': ('counter', s1[best_single][0]),
           'counter combo': ('counter', combo), best_latent: ('latent', LATENT[best_latent])}
conf = {}
for name, (up, hp) in CONFIRM.items():
    vals = [train4(up, hp, S1_STEPS, seed=sd)[1]['val'][-1] for sd in [0, 1, 2]]
    conf[name] = vals
    print(f'{name:28s} val {np.mean(vals):.4f} ± {np.std(vals):.4f}  {[round(v, 4) for v in vals]}')
BEST_NAME = min((n for n in conf if n.startswith('counter')), key=lambda n: np.mean(conf[n]))
BEST_HP = CONFIRM[BEST_NAME][1]
print('\nbest counter setting:', BEST_NAME, BEST_HP)

## Stage 2: toy conversion (pretrained fp model → ternary)

This mirrors the SmolLM2 experiment at toy scale: train a full-precision model, then convert it.
- **0**: the fp model. **0b**: naive rounding, no training.
- **B**: latent weights + Adam, initialized from the fp weights, with λ warmup (Hugging Face recipe).
- **C**: int8 counter, κ = 0 (counters start at zero), 0.5 or 1 (counters start partway to flipping, from the fp leftovers).

In [ ]:
QUANT['on'] = False
fp_model, Hfp = train4('latent', dict(lr=3e-3), 2000)
FP_STATE = {k: v.detach().clone() for k, v in fp_model.state_dict().items()}
print(f'fp pretraining: val {Hfp["val"][-1]:.4f}  ({Hfp["time"]:.0f}s)')
QUANT['on'] = True
naive = make_model('latent'); naive.load_state_dict(FP_STATE)
CONV_STEPS = 1000
s2 = {'0 fp model': Hfp['val'][-1], '0b naive rounding': evaluate(naive)}
print(f'0b naive rounding: val {s2["0b naive rounding"]:.4f}')
ARMS2 = {
    'B latent lr1e-3 lam200':  ('latent', dict(lr=1e-3), 200),
    'B latent lr3e-4 lam200':  ('latent', dict(lr=3e-4), 200),
    'B latent lr1e-3 no lam':  ('latent', dict(lr=1e-3), 0),
    'C counter kappa 0':       ('counter', {**BEST_HP, 'kappa': 0.0}, 200),
    'C counter kappa 0.5':     ('counter', {**BEST_HP, 'kappa': 0.5}, 200),
    'C counter kappa 1':       ('counter', {**BEST_HP, 'kappa': 1.0}, 200),
}
curves2 = {}
for name, (up, hp, ls) in ARMS2.items():
    _, H = train4(up, hp, CONV_STEPS, init=FP_STATE, lam_steps=ls, eval_every=100); s2[name] = H['val'][-1]; curves2[name] = H
    print(f'{name:26s} val {H["val"][-1]:.4f}  after 100 steps {H["val"][1]:.4f}  flips/step {np.mean(H["flips"][1:])*100:.3f}%')
BEST_KAPPA = min([0.0, 0.5, 1.0], key=lambda k: s2[f'C counter kappa {k:g}'])
print('best kappa:', BEST_KAPPA)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, (hp, H) in s1.items():
    axes[0].plot(H['step'], H['val'], ls='--' if name.startswith('latent') else '-', lw=2 if name in (BEST_NAME.replace('counter ', ''), 'base (tau 32)') else 0.8, label=name)
axes[0].set_title('Stage 1: from scratch'); axes[0].set_ylim(1.8, 2.7)
for name, H in curves2.items():
    axes[1].plot(H['step'], H['val'], ls='--' if name.startswith('B') else '-', label=name)
axes[1].axhline(s2['0 fp model'], color='k', lw=.8, label='fp model')
axes[1].set_title('Stage 2: conversion of a pretrained fp model')
for ax in axes: ax.set_xlabel('step'); ax.set_ylabel('val loss'); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.show()

## Stage 3: SmolLM2-135M pilot with the real training script

The next cell writes `train_bitnet.py`, the script for the rented GPU. Here it runs at a small budget on the two T4s, which:
- checks the script end to end (model replacement, fp16 grad scaling, checkpoints, evaluation);
- measures T4 throughput and memory;
- gives a first real comparison of the Hugging Face recipe (B) and the counter (C) on an actual pretrained model.

The T4 has no bf16, so the script uses fp16 with a grad scaler; on an A100 it switches to bf16 automatically.

In [ ]:
%%writefile /kaggle/working/train_bitnet.py
#!/usr/bin/env python
"""Convert a pretrained causal LM to ternary (BitNet b1.58 style) and compare two update rules.

Arms
  fp       evaluate the original model, no training (arm 0)
  latent   Hugging Face 1.58-bit recipe: fp32 latent weights + AdamW + STE + lambda warmup (arm B)
           with --tokens 0 it evaluates naive rounding of the pretrained weights (arm 0b)
  counter  int8 flip counter on the ternary weights: no full-precision linear weights at all (arm C)

Typical use
  python train_bitnet.py --prepare 6e7 --data data                      # tokenize FineWeb-edu
  python train_bitnet.py --arm latent --tokens 3e7 --data data --out runs/latent
  python train_bitnet.py --arm counter --tokens 3e7 --data data --out runs/counter --tau 32
Re-running the same command resumes from runs/<arm>/ckpt.pt.
"""
import argparse, json, math, os, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument('--arm', choices=['fp', 'latent', 'counter'])
    p.add_argument('--model', default='HuggingFaceTB/SmolLM2-135M')
    p.add_argument('--data', default='data')
    p.add_argument('--out', default=None)
    # data preparation
    p.add_argument('--prepare', type=float, default=0, help='tokenize this many training tokens and exit')
    p.add_argument('--val_tokens', type=float, default=2e6)
    p.add_argument('--subset', default='sample-10BT')
    # budget and batch
    p.add_argument('--tokens', type=float, default=3e7, help='training tokens (0 = evaluate only)')
    p.add_argument('--seq', type=int, default=1024)
    p.add_argument('--micro_bs', type=int, default=4)
    p.add_argument('--batch_tokens', type=int, default=65536)
    # optimisation
    p.add_argument('--lr', type=float, default=3e-4, help='lr of latent weights (latent arm)')
    p.add_argument('--fp_lr', type=float, default=3e-4, help='lr of embeddings, norms and ternary scales')
    p.add_argument('--sched', choices=['const', 'cos'], default='cos')
    p.add_argument('--warmup', type=float, default=0.02, help='lr warmup fraction')
    p.add_argument('--lambda_steps', type=int, default=200, help='quantization warmup steps (0 = off)')
    p.add_argument('--clip', type=float, default=1.0)
    p.add_argument('--extra_norm', type=int, default=0, help='RMSNorm before activation quant (changes the pretrained function)')
    # counter
    p.add_argument('--tau', type=float, default=32)
    p.add_argument('--kappa', type=float, default=0.5, help='counter init from the fp residual (0 = start at zero)')
    p.add_argument('--reset', choices=['zero', 'sub'], default='zero')
    p.add_argument('--tau_anneal', type=float, default=1.0, help='tau multiplier reached at the end of training')
    # runtime
    p.add_argument('--eval_every', type=int, default=100)
    p.add_argument('--eval_tokens', type=float, default=1e6)
    p.add_argument('--log_every', type=int, default=10)
    p.add_argument('--ckpt_every_min', type=float, default=20)
    p.add_argument('--max_minutes', type=float, default=0, help='checkpoint and stop after this many minutes')
    p.add_argument('--compile', type=int, default=1)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--price_per_hour', type=float, default=0)
    return p.parse_args()


# ----------------------------------------------------------------------------------------- data
def prepare(args):
    from datasets import load_dataset
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(args.model)
    eos = tok.eos_token_id
    assert len(tok) < 65535, 'uint16 storage needs a vocabulary below 65535'
    os.makedirs(args.data, exist_ok=True)
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name=args.subset, split='train', streaming=True)
    it = iter(ds)
    for split, n in [('val', int(args.val_tokens)), ('train', int(args.prepare))]:
        arr = np.memmap(os.path.join(args.data, f'{split}.bin'), dtype=np.uint16, mode='w+', shape=(n,))
        pos, t0, last = 0, time.time(), 0
        while pos < n:
            texts = [next(it)['text'] for _ in range(512)]
            for ids in tok(texts, add_special_tokens=False)['input_ids']:
                ids = ids + [eos]
                k = min(len(ids), n - pos)
                arr[pos:pos + k] = ids[:k]
                pos += k
                if pos >= n:
                    break
            if pos - last >= 5_000_000 or pos >= n:
                last = pos
                print(f'{split}: {pos / 1e6:.1f}M / {n / 1e6:.1f}M tokens  ({pos / (time.time() - t0) / 1e3:.0f}k tok/s)', flush=True)
        arr.flush()
    json.dump({'model': args.model, 'subset': args.subset, 'train_tokens': int(args.prepare),
               'val_tokens': int(args.val_tokens)}, open(os.path.join(args.data, 'meta.json'), 'w'))
    print('done')


def load_bin(path):
    return np.memmap(path, dtype=np.uint16, mode='r')


def get_micro(arr, g, B, T, device):
    span = B * T
    start = (g * span) % (len(arr) - span - 1)
    x = torch.from_numpy(arr[start:start + span].astype(np.int64)).view(B, T)
    return x.to(device, non_blocking=True)


# ---------------------------------------------------------------------------------------- model
def rms(x):
    return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6).to(x.dtype)


def act_quant(x, lam):
    # per-token absmax int8, blended in by lambda (STE: identity gradient)
    s = 127.0 / x.detach().abs().amax(-1, keepdim=True).float().clamp(min=1e-5)
    q = ((x.detach().float() * s).round().clamp(-128, 127) / s).to(x.dtype)
    return x + (lam.to(x.dtype) * (q - x.detach()))


def make_ternary_fn():
    fwd = torch.amp.custom_fwd(device_type='cuda')
    bwd = torch.amp.custom_bwd(device_type='cuda')

    class TernaryMM(torch.autograd.Function):
        """y = x @ (beta * t)^T. The weight gradient is routed to `sink`, a placeholder
        parameter, because the int8 ternary tensor itself cannot hold a gradient."""
        @staticmethod
        @fwd
        def forward(ctx, xq, t, beta, sink):
            ctx.save_for_backward(xq, t, beta)
            return F.linear(xq, t.to(beta.dtype) * beta)

        @staticmethod
        @bwd
        def backward(ctx, go):
            xq, t, beta = ctx.saved_tensors
            w = t.to(beta.dtype) * beta
            gx = go @ w.to(go.dtype)
            gW = go.reshape(-1, go.shape[-1]).t() @ xq.reshape(-1, xq.shape[-1]).to(go.dtype)
            gbeta = (gW.float() * t.float()).sum().reshape(beta.shape)
            return gx.to(xq.dtype), None, gbeta, gW.float()
    return TernaryMM


class BitLinear(nn.Module):
    def __init__(self, lin, arm, lam, args, fn):
        super().__init__()
        assert lin.bias is None, 'bias not supported'
        W = lin.weight.data.float()
        self.arm, self.extra_norm, self.fn = arm, bool(args.extra_norm), fn
        self.register_buffer('lam', lam, persistent=False)     # one 0-d tensor shared by every layer
        if arm == 'latent':
            self.weight = nn.Parameter(W.clone())
        else:
            beta = W.abs().mean().clamp(min=1e-5)
            u = W / beta
            t = u.round().clamp(-1, 1)
            r = (u - t).clamp(-0.5, 0.5)
            lim = max(1.0, args.tau - 1)
            c = (args.kappa * 2 * r * args.tau).round().clamp(-lim, lim).clamp(-127, 127)
            self.register_buffer('t', t.to(torch.int8))
            self.register_buffer('c', c.to(torch.int8))
            self.beta = nn.Parameter(beta.reshape(1))
            self.sink = nn.Parameter(torch.zeros_like(W))       # receives dL/dW_eff

    def ternary(self):
        if self.arm == 'counter':
            return self.t
        W = self.weight.detach()
        return (W / W.abs().mean().clamp(min=1e-5)).round().clamp(-1, 1).to(torch.int8)

    def forward(self, x):
        if self.extra_norm:
            x = rms(x)
        xq = act_quant(x, self.lam)
        if self.arm == 'latent':
            W = self.weight
            beta = W.detach().abs().mean().clamp(min=1e-5)
            q = (W.detach() / beta).round().clamp(-1, 1) * beta
            w = W + (self.lam * (q - W.detach()))
            return F.linear(xq, w)
        return self.fn.apply(xq, self.t, self.beta, self.sink)


def build(args, device):
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.float32,
                                                 attn_implementation='sdpa').to(device)
    model.config.use_cache = False
    if args.arm == 'fp':
        return model, [], None
    lam = torch.ones((), device=device)
    fn = make_ternary_fn()
    bls = []
    for name, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname != 'lm_head':
                bl = BitLinear(child, args.arm, lam, args, fn)
                setattr(mod, cname, bl)
                bls.append((f'{name}.{cname}', bl))
    torch.cuda.empty_cache()
    return model, bls, lam


@torch.no_grad()
def counter_step(bl, G, tau, reset):
    s = -G / G.pow(2).mean().sqrt().clamp(min=1e-30)
    inc = torch.floor(s + torch.rand_like(s))                  # unbiased stochastic rounding
    cf = (bl.c.float() + inc).clamp(-127, 127)
    t = bl.t.float()
    up, dn = cf >= tau, cf <= -tau
    nt = torch.where(up, (t + 1).clamp(max=1), torch.where(dn, (t - 1).clamp(min=-1), t))
    if reset == 'zero':
        cf = torch.where(up | dn, torch.zeros_like(cf), cf)
    else:
        cf = torch.where(up, cf - tau, torch.where(dn, cf + tau, cf))
    flips = (nt != t).sum()
    bl.c.copy_(cf.round().to(torch.int8))
    bl.t.copy_(nt.to(torch.int8))
    return flips


@torch.no_grad()
def evaluate(model, val, args, device, amp, lam):
    old = None
    if lam is not None:
        old = lam.item(); lam.fill_(1.0)                       # always evaluate the fully ternary model
    model.eval()
    T, B = args.seq, args.micro_bs
    nseq = int(min(len(val) - 1, args.eval_tokens)) // T
    tot = n = 0
    for i in range(0, nseq, B):
        b = min(B, nseq - i)
        x = torch.from_numpy(val[i * T:(i + b) * T].astype(np.int64)).view(b, T).to(device)
        with torch.autocast('cuda', dtype=amp):
            loss = model(input_ids=x, labels=x).loss
        tot += loss.item() * b; n += b
    model.train()
    if lam is not None:
        lam.fill_(old)
    return tot / max(n, 1)


def save_ckpt(path, model, opt, scaler, step, extra):
    sd = {k: v for k, v in model.state_dict().items() if not k.endswith('.sink')}
    tmp = path + '.tmp'
    torch.save({'model': sd, 'opt': opt.state_dict() if opt else None, 'scaler': scaler.state_dict(),
                'step': step, 'cpu_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state(),
                **extra}, tmp)
    os.replace(tmp, path)


# ----------------------------------------------------------------------------------------- main
def main():
    args = get_args()
    if args.prepare:
        return prepare(args)
    assert args.arm and args.out, '--arm and --out are required'
    os.makedirs(args.out, exist_ok=True)
    assert torch.cuda.is_available(), 'needs a CUDA GPU'
    device = 'cuda'
    cap = torch.cuda.get_device_capability()
    amp = torch.bfloat16 if cap[0] >= 8 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.manual_seed(args.seed)
    print(f'GPU {torch.cuda.get_device_name(0)} | autocast {amp} | arm {args.arm}', flush=True)

    model, bls, lam = build(args, device)
    val = load_bin(os.path.join(args.data, 'val.bin'))
    log_f = open(os.path.join(args.out, 'log.jsonl'), 'a')
    def log(d):
        log_f.write(json.dumps(d) + '\n'); log_f.flush()

    n_micro = max(1, args.batch_tokens // (args.micro_bs * args.seq))
    tok_step = n_micro * args.micro_bs * args.seq
    steps = int(args.tokens // tok_step)

    if steps == 0 or args.arm == 'fp':
        v = evaluate(model, val, args, device, amp, lam)
        res = dict(arm=args.arm, tokens=0, val_loss=v, ppl=math.exp(v))
        print(json.dumps(res), flush=True)
        json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'))
        return

    train = load_bin(os.path.join(args.data, 'train.bin'))
    if steps * tok_step > len(train):
        print(f'WARNING: {steps * tok_step / 1e6:.0f}M training tokens requested but only {len(train) / 1e6:.0f}M prepared: data will repeat', flush=True)

    latent_ws = [bl.weight for _, bl in bls if bl.arm == 'latent']
    sinks = [bl.sink for _, bl in bls if bl.arm == 'counter']
    skip = {id(p) for p in latent_ws + sinks}
    others = [p for p in model.parameters() if p.requires_grad and id(p) not in skip]
    groups = [{'params': others, 'lr': args.fp_lr}]
    if latent_ws:
        groups.append({'params': latent_ws, 'lr': args.lr})
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), weight_decay=0.0)
    opt2 = torch.optim.SGD(sinks, lr=0.0) if sinks else None    # lets the grad scaler see inf in sinks
    base_lrs = [g['lr'] for g in opt.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=(amp == torch.float16))

    def lr_mult(s):
        w = max(1, int(args.warmup * steps))
        if s < w:
            return (s + 1) / w
        if args.sched == 'const':
            return 1.0
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, steps - w)))

    start = 0
    ck = os.path.join(args.out, 'ckpt.pt')
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(st['model'], strict=False)
        opt.load_state_dict(st['opt']); scaler.load_state_dict(st['scaler'])
        torch.set_rng_state(st['cpu_rng'].cpu()); torch.cuda.set_rng_state(st['cuda_rng'].cpu())
        start = st['step']
        print(f'resumed at step {start}', flush=True)
    if start >= steps:
        print('already finished'); return

    model.train()
    cmodel = torch.compile(model) if args.compile else model
    if args.compile:
        # the quantization warmup changes the shared `lam` buffer in place; check the compiled
        # model really reads it at run time instead of baking in one value
        x = get_micro(train, 0, args.micro_bs, args.seq, device)
        try:
            with torch.no_grad(), torch.autocast('cuda', dtype=amp):
                lam.fill_(0.0); a = cmodel(input_ids=x).logits.float().mean().item()
                lam.fill_(1.0); b = cmodel(input_ids=x).logits.float().mean().item()
            if a == b:
                print('WARNING: compiled model ignores lambda changes; running without torch.compile', flush=True)
                cmodel = model
        except Exception as e:
            print(f'torch.compile failed ({type(e).__name__}: {str(e)[:200]}); running without it', flush=True)
            cmodel = model
    lin_w = sum((bl.weight.numel() if bl.arm == 'latent' else bl.t.numel()) for _, bl in bls)
    state_bytes = 12 if args.arm == 'latent' else 2
    print(f'{steps} steps x {tok_step} tokens | ternary weights {lin_w / 1e6:.1f}M | '
          f'persistent training state per ternary weight: {state_bytes} bytes '
          f'({"fp32 latent + 2 Adam moments" if args.arm == "latent" else "int8 ternary + int8 counter"})', flush=True)

    prev_t = {n: bl.ternary().clone() for n, bl in bls} if args.arm == 'latent' else None
    t_start, t_ck = time.time(), time.time()
    flips_acc, loss_acc, n_acc = 0, 0.0, 0
    tput = None
    for step in range(start, steps):
        frac = step / steps
        if lam is not None:
            lam.fill_(min(1.0, (step + 1) / args.lambda_steps) if args.lambda_steps else 1.0)
        for g, b in zip(opt.param_groups, base_lrs):
            g['lr'] = b * lr_mult(step)
        if step == start + 3:
            torch.cuda.synchronize(); t_tp = time.time()
        for j in range(n_micro):
            x = get_micro(train, step * n_micro + j, args.micro_bs, args.seq, device)
            with torch.autocast('cuda', dtype=amp):
                try:
                    loss = cmodel(input_ids=x, labels=x).loss / n_micro
                except Exception as e:
                    if cmodel is model:
                        raise
                    print(f'torch.compile failed ({type(e).__name__}: {str(e)[:200]}); continuing without it', flush=True)
                    cmodel = model
                    loss = cmodel(input_ids=x, labels=x).loss / n_micro
            scaler.scale(loss).backward()
            loss_acc += loss.detach().float()
        n_acc += 1
        scaler.unscale_(opt)
        if opt2:
            scaler.unscale_(opt2)
        if args.clip:
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g['params']], args.clip)
        bad = None
        if sinks:
            bad = torch.zeros((), dtype=torch.bool, device=device)
            for s_ in sinks:
                bad |= ~torch.isfinite(s_.grad).all()
        scaler.step(opt)
        if opt2:
            scaler.step(opt2)
        scaler.update()
        if sinks and not bad.item():
            tau = args.tau * (1 + (args.tau_anneal - 1) * frac)
            for _, bl in bls:
                flips_acc += counter_step(bl, bl.sink.grad.float(), tau, args.reset)
        opt.zero_grad(set_to_none=True)
        if opt2:
            opt2.zero_grad(set_to_none=True)

        if step == start + 13:
            torch.cuda.synchronize()
            tput = 10 * tok_step / (time.time() - t_tp)
            h1b = 1e9 / tput / 3600
            msg = f'THROUGHPUT {tput / 1e3:.1f}k tokens/s | {h1b:.1f} GPU-hours per 1B tokens | peak memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB'
            if args.price_per_hour:
                msg += f' | ${args.price_per_hour * h1b:.2f} per 1B tokens'
            msg += f' | this run: {(steps - step) * tok_step / tput / 3600:.1f} h left'
            print(msg, flush=True)
            log(dict(kind='throughput', tok_s=tput, hours_per_1B=h1b, peak_gib=torch.cuda.max_memory_allocated() / 2**30))

        done = step + 1
        if done % args.log_every == 0 or done == steps:
            if args.arm == 'latent':
                nf = 0
                for n, bl in bls:
                    t = bl.ternary(); nf += (t != prev_t[n]).sum().item(); prev_t[n] = t
                flips = nf / lin_w / n_acc
            else:
                flips = float(flips_acc) / lin_w / n_acc
            rec = dict(kind='train', step=done, tokens=done * tok_step, loss=float(loss_acc) / n_acc,
                       lr=opt.param_groups[0]['lr'], lam=float(lam.item()), flips=flips,
                       scale=scaler.get_scale() if scaler.is_enabled() else 1.0, elapsed=time.time() - t_start)
            log(rec)
            print(f"step {done}/{steps} tokens {done * tok_step / 1e6:.1f}M loss {rec['loss']:.4f} "
                  f"flips/step {flips * 100:.4f}% lam {rec['lam']:.2f} {rec['elapsed'] / 60:.1f} min", flush=True)
            flips_acc, loss_acc, n_acc = 0, 0.0, 0

        if done % args.eval_every == 0 or done == steps:
            v = evaluate(model, val, args, device, amp, lam)
            zeros = sum((bl.ternary() == 0).sum().item() for _, bl in bls) / lin_w
            log(dict(kind='eval', step=done, tokens=done * tok_step, val_loss=v, ppl=math.exp(v), zeros=zeros))
            print(f'EVAL step {done} tokens {done * tok_step / 1e6:.1f}M val_loss {v:.4f} ppl {math.exp(v):.2f} zeros {zeros * 100:.1f}%', flush=True)

        stop = args.max_minutes and (time.time() - t_start) / 60 > args.max_minutes
        if done == steps or stop or (time.time() - t_ck) / 60 > args.ckpt_every_min:
            save_ckpt(ck, model, opt, scaler, done, {})
            t_ck = time.time()
        if stop:
            print(f'stopped after {args.max_minutes} min at step {done}; re-run the same command to resume', flush=True)
            return

    v = evaluate(model, val, args, device, amp, lam)
    res = dict(arm=args.arm, tokens=steps * tok_step, val_loss=v, ppl=math.exp(v), tok_s=tput,
               peak_gib=torch.cuda.max_memory_allocated() / 2**30, state_bytes_per_weight=state_bytes,
               args=vars(args))
    json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'), indent=1)
    sd = {k: v_ for k, v_ in model.state_dict().items() if not k.endswith('.sink')}
    torch.save(sd, os.path.join(args.out, 'final.pt'))
    print('FINAL', json.dumps({k: res[k] for k in ['arm', 'tokens', 'val_loss', 'ppl']}), flush=True)


if __name__ == '__main__':
    main()

### Tokenize FineWeb-edu (2M validation + 60M training tokens; about 5–10 min)

In [ ]:
import subprocess, sys
W = '/kaggle/working'
DATA = f'{W}/data'
if not os.path.exists(f'{DATA}/train.bin'):
    subprocess.run([sys.executable, f'{W}/train_bitnet.py', '--prepare', '6e7', '--data', DATA], check=True)
print(open(f'{DATA}/meta.json').read())

### Arms 0 and 0b: the fp model and naive rounding (evaluation only)

In [ ]:
for arm, out in [('fp', 'runs/fp'), ('latent', 'runs/naive')]:
    subprocess.run([sys.executable, f'{W}/train_bitnet.py', '--arm', arm, '--tokens', '0', '--data', DATA,
                    '--out', f'{W}/{out}', '--micro_bs', '8'], check=True)

### Arms B and C in parallel, one per T4

Uses the best counter setting from stages 1–2. `PILOT_TOKENS = 3e7` per arm takes about 1–1.5 h on a T4; the script prints its measured throughput and the time left. Re-running this cell resumes from checkpoints.

In [ ]:
PILOT_TOKENS = '3e7'
common = ['--tokens', PILOT_TOKENS, '--data', DATA, '--micro_bs', '4', '--seq', '1024', '--batch_tokens', '65536',
          '--eval_every', '50', '--max_minutes', '600']
tau = str(BEST_HP.get('tau', 32)); anneal = str(BEST_HP.get('anneal', 1.0)); reset = BEST_HP.get('reset', 'zero')
jobs = {
    'latent':  common + ['--arm', 'latent', '--lr', '3e-4', '--out', f'{W}/runs/latent'],
    'counter': common + ['--arm', 'counter', '--tau', tau, '--tau_anneal', anneal, '--reset', reset,
                         '--kappa', str(BEST_KAPPA), '--out', f'{W}/runs/counter'],
}
extra = {k: v for k, v in BEST_HP.items() if k not in ('tau', 'anneal', 'reset', 'kappa')}
if extra: print('note: train_bitnet.py does not implement', extra, '- the pilot uses tau/anneal/reset only')
ngpu = torch.cuda.device_count()
procs = {}
for i, (name, a) in enumerate(jobs.items()):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % ngpu))
    procs[name] = subprocess.Popen([sys.executable, f'{W}/train_bitnet.py'] + a, env=env,
                                   stdout=open(f'{W}/{name}.out', 'w'), stderr=subprocess.STDOUT)
    if ngpu == 1: procs[name].wait()          # one GPU: run one after the other
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for name in procs:
        lines = [l for l in open(f'{W}/{name}.out').read().splitlines() if l.strip()]
        print(f'[{name}]', lines[-1] if lines else '(starting)')
for name in procs:
    print(f'--- {name} ---'); print('\n'.join([l for l in open(f'{W}/{name}.out').read().splitlines()
                                              if l.startswith(('GPU', 'THROUGHPUT', 'EVAL', 'FINAL', 'WARNING', 'torch.compile', 'stopped'))]))

## Summary

In [ ]:
def read_log(path):
    if not os.path.exists(path): return []
    return [json.loads(l) for l in open(path)]
fig, ax = plt.subplots(figsize=(7, 4))
for name in ['latent', 'counter']:
    ev = [r for r in read_log(f'{W}/runs/{name}/log.jsonl') if r['kind'] == 'eval']
    if ev: ax.plot([r['tokens'] / 1e6 for r in ev], [r['val_loss'] for r in ev], marker='o', label=name)
for name, ls in [('fp', '-'), ('naive', ':')]:
    p = f'{W}/runs/{name}/result.json'
    if os.path.exists(p): ax.axhline(json.load(open(p))['val_loss'], color='k', ls=ls, lw=.8, label=f'{name} (eval only)')
ax.set_xlabel('training tokens (M)'); ax.set_ylabel('FineWeb-edu val loss'); ax.set_title('SmolLM2-135M → ternary'); ax.grid(alpha=.3); ax.legend()
plt.show()

print('STAGE 1 screen:'); [print(f'  {n:22s} {H["val"][-1]:.4f}') for n, (hp, H) in s1.items()]
print('STAGE 1 confirm (3 seeds):'); [print(f'  {n:28s} {np.mean(v):.4f} ± {np.std(v):.4f}') for n, v in conf.items()]
print('best counter:', BEST_HP, '| best kappa:', BEST_KAPPA)
print('STAGE 2 conversion:'); [print(f'  {n:26s} {v:.4f}') for n, v in s2.items()]
print('STAGE 3 pilot:')
for name in ['fp', 'naive', 'latent', 'counter']:
    p = f'{W}/runs/{name}/result.json'
    if os.path.exists(p):
        r = json.load(open(p)); print(f"  {name:8s} val {r['val_loss']:.4f} ppl {r['ppl']:.2f} tokens {r['tokens']/1e6:.0f}M "
                                       f"tok/s {r.get('tok_s') or 0:.0f} peak {r.get('peak_gib') or 0:.1f} GiB")
    tp = [r for r in read_log(f'{W}/runs/{name}/log.jsonl') if r.get('kind') == 'throughput']
    if tp: print(f'    throughput {tp[-1]["tok_s"]/1e3:.1f}k tok/s, {tp[-1]["hours_per_1B"]:.1f} T4-hours per 1B tokens')
json.dump({'s1': {n: H['val'] for n, (hp, H) in s1.items()}, 'confirm': conf, 'best_hp': BEST_HP, 'best_kappa': BEST_KAPPA,
           's2': s2}, open(f'{W}/v4_results.json', 'w'))